In [7]:
from pathlib import Path
import numpy as np
import tensorflow as tf

PROJECT_DIR = Path.cwd()
while not (PROJECT_DIR / "inception-v1" / "saved_model.pb").is_file():
    parent = PROJECT_DIR.parent
    if parent == PROJECT_DIR:
        raise FileNotFoundError("Impossible de localiser le dossier inception-v1.")
    PROJECT_DIR = parent

MODEL_PATH = PROJECT_DIR / "inception-v1"

print("TensorFlow version:", tf.__version__)

model = tf.saved_model.load(str(MODEL_PATH))

print("Model loaded successfully!")
print("Available signatures:")
print(model.signatures.keys())

TensorFlow version: 2.22.0-rc0
Model loaded successfully!
Available signatures:
KeysView(_SignatureMap({'serving_default': <ConcreteFunction (*, inputs: TensorSpec(shape=(None, None, None, 3), dtype=tf.float32, name='inputs')) -> Dict[['logits', TensorSpec(shape=(None, 1001), dtype=tf.float32, name='logits')]] at 0x7F28F01AF590>}))


In [8]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

DATA_DIR = PROJECT_DIR / "data" / "classification"
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
VALIDATION_SPLIT = 0.30

train_ds, temporary_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    subset="both",
    seed=SEED,
    shuffle=True,
)

supported_extensions = {".jpeg", ".jpg", ".png", ".bmp", ".gif"}
image_count = sum(
    1
    for class_dir in DATA_DIR.iterdir()
    if class_dir.is_dir()
    for image_path in class_dir.iterdir()
    if image_path.is_file() and image_path.suffix.lower() in supported_extensions
)
temporary_count = int(image_count * VALIDATION_SPLIT)
validation_count = temporary_count // 2

temporary_ds = temporary_ds.unbatch().shuffle(
    buffer_size=temporary_count,
    seed=SEED,
    reshuffle_each_iteration=False,
)
validation_ds = temporary_ds.take(validation_count).batch(BATCH_SIZE)
test_ds = temporary_ds.skip(validation_count).batch(BATCH_SIZE)

def preprocess(images, labels):
    images = tf.cast(images, tf.float32) / 127.5 - 1.0
    return images, labels

train_ds = train_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
validation_ds = validation_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
validation_ds = validation_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

print("Classes:", CLASS_NAMES)
print("Images:", image_count)
print("Approximate split: 70% train, 15% validation, 15% test")

Found 7023 files belonging to 4 classes.
Using 4917 files for training.
Using 2106 files for validation.
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Images: 7023
Approximate split: 70% train, 15% validation, 15% test


In [9]:
backbone = tf.keras.layers.TFSMLayer(
    str(MODEL_PATH),
    call_endpoint="serving_default",
    trainable=False,
)

inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name="brain_mri")
pretrained_logits = backbone(inputs)["logits"]
outputs = tf.keras.layers.Dense(
    len(CLASS_NAMES),
    activation="softmax",
    name="tumor_classification_head",
)(pretrained_logits)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name="googlenet_tumor_classifier")

LEARNING_RATE = 1e-3
EPOCHS = 20
LOSS = tf.keras.losses.SparseCategoricalCrossentropy()
OPTIMIZER = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

model.compile(
    optimizer=OPTIMIZER,
    loss=LOSS,
    metrics=["accuracy"],
)

model.summary()
print(f"Optimizer: Adam | learning rate: {LEARNING_RATE} | loss: sparse categorical crossentropy")

Model: "googlenet_tumor_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ brain_mri (InputLayer)          │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ tfsm_layer_1 (TFSMLayer)        │ (None, 1001)           │     6,633,209 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ tumor_classification_head       │ (None, 4)              │         4,008 │
│ (Dense)                         │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,637,217 (25.32 MB)

 Trainable params: 4,008 (15.66 KB)

 Non-trainable params: 6,633,209 (25.30 MB)

Optimizer: Adam | learning rate: 0.001 | loss: sparse categorical crossentropy


In [ ]:
checkpoint_path = PROJECT_DIR / "models/inception_v1_tumor_classifier.keras"

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor="val_loss",
        save_best_only=True,
    ),
]

history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

print("Best model saved to:", checkpoint_path)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
y_probabilities = model.predict(test_ds, verbose=0)
y_predicted = np.argmax(y_probabilities, axis=1)

precision, recall, f1_score, _ = precision_recall_fscore_support(
    y_true,
    y_predicted,
    average="macro",
    zero_division=0,
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {accuracy_score(y_true, y_predicted):.4f}")
print(f"Test precision (macro): {precision:.4f}")
print(f"Test recall (macro): {recall:.4f}")
print(f"Test F1-score (macro): {f1_score:.4f}")
print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_predicted,
        labels=range(len(CLASS_NAMES)),
        target_names=CLASS_NAMES,
        zero_division=0,
    )
)
print("Confusion matrix:")
print(confusion_matrix(y_true, y_predicted, labels=range(len(CLASS_NAMES))))

W0000 00:00:1793178654.290211    1375 cpu_allocator_impl.cc:174] Allocation of 102760448 exceeds 10% of free system memory.
W0000 00:00:1793178654.595038    1375 cpu_allocator_impl.cc:174] Allocation of 77070336 exceeds 10% of free system memory.
W0000 00:00:1793178654.946160    1371 cpu_allocator_impl.cc:174] Allocation of 48168960 exceeds 10% of free system memory.
W0000 00:00:1793178655.249953    1373 cpu_allocator_impl.cc:174] Allocation of 102760448 exceeds 10% of free system memory.
W0000 00:00:1793178655.375877    1373 cpu_allocator_impl.cc:174] Allocation of 77070336 exceeds 10% of free system memory.


Test loss: 1.9003
Test accuracy: 0.2545
Test precision (macro): 0.1985
Test recall (macro): 0.2530
Test F1-score (macro): 0.1753

Classification report:
              precision    recall  f1-score   support

      glioma       0.00      0.00      0.00       240
  meningioma       0.24      0.75      0.36       254
     notumor       0.35      0.22      0.27       303
   pituitary       0.20      0.04      0.07       256

    accuracy                           0.25      1053
   macro avg       0.20      0.25      0.18      1053
weighted avg       0.21      0.25      0.18      1053

Confusion matrix:
[[  0 199  20  21]
 [  5 191  51   7]
 [ 12 213  67  11]
 [  2 190  54  10]]
